<h3>실습 안내</h3><p>수업을 위해 만든 PDF <code>E01.pdf</code>를 직접 읽습니다. PDF 한 쪽이 본문과 출처 정보를 담은 Document가 되고, 다시 작은 검색 조각인 청크로 나뉘는 과정을 확인합니다.</p>
<h3>호출 표기와 반환값 읽기</h3><p><code>reader.pages[1]</code>은 0부터 세는 목록의 두 번째 쪽을 고르고, <code>extract_text()</code>는 그 쪽을 문자열로 반환합니다. <code>split_documents([document])</code>는 Document 목록을 받아 더 짧은 Document 목록을 반환하며 이때 임베딩이나 검색은 실행되지 않습니다.</p><p>문자열만 반환하는 <code>split_text()</code>와 달리 <code>split_documents()</code>는 기존 메타데이터를 청크에 복사합니다. 조항을 추적해야 하는 문서는 분할 전에 조항 ID도 메타데이터에 넣습니다.</p>

In [1]:
# PDF 텍스트 추출과 Document 분할에 필요한 패키지를 설치합니다.
# 이 절은 외부 API를 호출하지 않으므로 .env를 읽지 않습니다.
%pip install -q "pypdf==6.17.0" "langchain==1.4.0" "langchain-text-splitters==1.1.2" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.

<h3>1. PDF 경로 찾기</h3><p>현재 폴더와 부모 폴더를 차례로 확인해 실습 PDF가 모여 있는 폴더를 찾습니다.</p>

In [2]:
# project_root는 labs/data/rag-corpus가 존재하는 첫 상위 폴더입니다.
# break는 파일을 찾은 즉시 for 반복을 종료합니다.
from pathlib import Path

start = Path.cwd().resolve()
project_root = None
folders = [start]
folders.extend(start.parents)
for folder in folders:
    if (folder / "labs/data/rag-corpus").exists():
        project_root = folder
        break
assert project_root is not None, "실습 말뭉치를 찾지 못했습니다."
pdf_path = project_root / "labs/data/rag-corpus/pdf/easy/E01.pdf"
assert pdf_path.exists(), f"PDF가 없습니다: {pdf_path}"
print("실습 파일:", pdf_path.relative_to(project_root))

실습 파일: labs/data/rag-corpus/pdf/easy/E01.pdf


<h3>2. PDF 한 쪽을 Document로 바꾸기</h3><p>정기 휴관일이 있는 2쪽을 고르고 본문·문서 ID·쪽·파일명을 함께 저장합니다.</p>

In [3]:
# reader.pages[1]의 인덱스 1은 PDF의 두 번째 쪽을 선택합니다.
# Document는 page_content에 본문, metadata에 출처 정보를 보관합니다.
from pypdf import PdfReader
from langchain_core.documents import Document

reader = PdfReader(pdf_path)
page_text = reader.pages[1].extract_text()
document = Document(
    page_content=page_text,
    metadata={"doc_id": "E01", "page": 2, "source": pdf_path.name},
)
print("Document 타입:", type(document).__name__)
print("metadata:", document.metadata)
print("content:", document.page_content[:240])

Document 타입: Document
metadata: {'doc_id': 'E01', 'page': 2, 'source': 'E01.pdf'}
content: 1. 도서관 소개
미르시립도서관은 1998년 10월 미르시 중앙동에 문을 연 공공도서관입니다. 지상 4층, 지하 1층 규모
이며 장서 42만 권과 전자책 6만 종을 보유하고 있습니다. 2024년 리모델링을 마치고 3층에 '메이커 스
페이스'와 '조용한 열람실'을 새로 열었습니다.
도서관의 사명은 "누구나, 언제나, 무료로"입니다. 시민 모두가 자료를 자유롭게 이용하고 배울 수 있도
록 돕는 것을 최우선으로 합니다.
2. 이용 


<h3>3. 기본 분할 실행</h3><p>청크 한 개의 최대 글자 수를 500으로, 이웃 청크가 함께 가지는 글자 수를 50으로 정합니다. 나눈 뒤 청크 개수·길이·출처 정보(metadata)를 출력합니다.</p>

In [4]:
# split_documents()는 Document 리스트를 받으므로 [document]로 감쌍니다.
# 반환된 chunks도 Document 리스트이며 원본 metadata가 복사됩니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents([document])
print("청크 수:", len(chunks))
for number, chunk in enumerate(chunks, start=1):
    print(number, "길이:", len(chunk.page_content), "metadata:", chunk.metadata)
    print(chunk.page_content[:100])

청크 수: 2
1 길이: 498 metadata: {'doc_id': 'E01', 'page': 2, 'source': 'E01.pdf'}
1. 도서관 소개
미르시립도서관은 1998년 10월 미르시 중앙동에 문을 연 공공도서관입니다. 지상 4층, 지하 1층 규모
이며 장서 42만 권과 전자책 6만 종을 보유하고 있습니
2 길이: 452 metadata: {'doc_id': 'E01', 'page': 2, 'source': 'E01.pdf'}
구역 평일 주말 비고
종합자료실(2층) 09:00~22:00 09:00~18:00
어린이자료실(1층) 09:00~18:00 09:00~18:00 보호자 동반 권장
디지털자료실(2층


<h3>4. 값 하나만 바꾸어 비교</h3><p><code>chunk_overlap=50</code>은 그대로 두고 <code>chunk_size</code>만 500에서 300으로 줄입니다.</p>

In [5]:
# smaller와 splitter의 차이는 chunk_size 하나뿐입니다.
# 기본·변경 결과의 개수와 첫 청크 길이를 같은 형식으로 비교합니다.
smaller = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
small_chunks = smaller.split_documents([document])
print("기본 500:", len(chunks), "개, 첫 길이", len(chunks[0].page_content))
print("변경 300:", len(small_chunks), "개, 첫 길이", len(small_chunks[0].page_content))
assert len(small_chunks) >= len(chunks)

기본 500: 2 개, 첫 길이 498
변경 300: 4 개, 첫 길이 242


<h3>5. 실제 청크의 토큰 수 확인</h3><p><code>text-embedding-3-small</code>이 사용하는 <code>cl100k_base</code> 토크나이저로 첫 청크를 인코딩합니다. Ch 5의 GPT-2·KLUE 토크나이저와 계산 단위가 다르므로 그때의 수치와 직접 비교하지 않습니다.</p>

In [6]:
# encoding_for_model()로 임베딩 모델에 맞는 토크나이저를 명시합니다.
# 같은 문자열도 토크나이저가 다르면 토큰 수가 달라질 수 있습니다.
import tiktoken

embedding_tokenizer = tiktoken.encoding_for_model("text-embedding-3-small")
first_chunk_text = chunks[0].page_content
first_chunk_tokens = embedding_tokenizer.encode(first_chunk_text)
print("토크나이저:", embedding_tokenizer.name)
print("첫 청크 글자 수:", len(first_chunk_text))
print("첫 청크 토큰 수:", len(first_chunk_tokens))
assert len(first_chunk_tokens) > 0

토크나이저: cl100k_base
첫 청크 글자 수: 498
첫 청크 토큰 수: 491


<h3>정리</h3><p>Document의 문서 ID와 쪽 번호(metadata)는 나눈 뒤에도 남았습니다. 이웃 청크가 함께 가지는 글자 수는 그대로 두고, 청크 최대 길이만 줄이자 청크 개수와 길이가 바뀌었습니다. 첫 청크의 글자 수와 임베딩 모델 토크나이저의 토큰 수는 서로 다른 값으로 확인했습니다.</p>

<p><strong>원인</strong> — 겹침이 없으면 청크 경계에 놓인 “월요일”이 두 조각으로 갈라져 어느 청크에도 온전히 남지 않을 수 있습니다.</p>

In [7]:
# 경계 확인용 Document에서 “월요일”이 정확히 청크 경계에 걸리게 만듭니다.
# any()는 어느 청크든 target 전체를 포함하는지 확인합니다.
boundary_text = "가나다라마바사아자차" * 3 + "정기 휴관일은 월요일입니다."
boundary_document = Document(page_content=boundary_text)
target = "월요일"
no_overlap_splitter = RecursiveCharacterTextSplitter(
    chunk_size=40, chunk_overlap=0, separators=[""]
)
before_chunks = no_overlap_splitter.split_documents([boundary_document])
before_texts = [chunk.page_content for chunk in before_chunks]
print("Before 청크:", before_texts)
try:
    assert any(target in text for text in before_texts)
except AssertionError:
    print("예상한 실패: 월요일이 청크 경계에서 갈라졌습니다.")
else:
    raise AssertionError("청크 경계 실패가 재현되어야 합니다.")

Before 청크: ['가나다라마바사아자차가나다라마바사아자차가나다라마바사아자차정기 휴관일은 월요', '일입니다.']
예상한 실패: 월요일이 청크 경계에서 갈라졌습니다.


<p><strong>수정</strong> — 다른 값은 유지하고 <code>chunk_overlap</code>만 0에서 5로 늘려 경계 양쪽 문자를 함께 보존합니다.</p>

In [8]:
# overlap_splitter는 앞의 설정에서 chunk_overlap만 5로 바꿉니다.
# 겹친 청크 중 하나에 “월요일” 전체가 들어왔는지 확인합니다.
overlap_splitter = RecursiveCharacterTextSplitter(
    chunk_size=40, chunk_overlap=5, separators=[""]
)
after_chunks = overlap_splitter.split_documents([boundary_document])
after_texts = [chunk.page_content for chunk in after_chunks]
print("After 청크:", after_texts)
assert any(target in text for text in after_texts)

After 청크: ['가나다라마바사아자차가나다라마바사아자차가나다라마바사아자차정기 휴관일은 월요', '일은 월요일입니다.']
